# Bench 实验
逐格运行，不要 Run All；标「可重跑」的格可以单独再跑。ESP 每次 reset 时保持线圈电源关闭。
结束后：`uv run python controller/control/safe_off.py`，关电源。步骤说明见 `README.md`。
本机 1280×800 @ 120 fps，预览左 A（1）、右 B（0），两机均使用原始朝向。可选 6A/9A 保留本地 axis_pose。


## 导入

In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'LLM/bench.py').exists())
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from IPython.display import display, Image
from LLM.bench import Bench

## 设置
改任何一项都从「建立 bench」重跑。`CALIBRATION = None` 表示重新标定；`PLUMB = None` 表示不做装置定位（sync check）。

In [ ]:
CAMERAS = (1, 0)  # 已核对预览：左 A = 设备 1，右 B = 设备 0
PORT = '/dev/cu.usbserial-0001'  # 本机 ESP 端口，运行前核对
MODE = (1280, 800, 120)  # 恢复本机原采集模式；实际 fps 由试录源时间验收
EXPOSURE_MS = None  # 默认自动曝光；需要手动时填毫秒数（120 fps 时小于 8.33 ms）
GAIN = 100  # 仅手动曝光模式使用
CALIBRATION = None  # 步骤 2 按当前机位和朝向重新标定
PLUMB = None  # 当前先做 sync check + axis_pose；无需吊住 robot
RAMP = 'two_stage_1to195Hz_hold5s'  # 无录像驱动、步骤 7 预览和步骤 8 实验共用这个选择
NOTE = ''  # 每组填写杆径、固定方式等实验条件
WARNING_REVIEW = ''  # 只写本次试录 warnings 的检查结论，不沿用历史确认
print('可选 ramp:', sorted(p.stem for p in (PROJECT/'LLM/ramps').glob('*.json')))


In [ ]:
import importlib
from LLM.record import drive_only
from LLM.record.ramps import ramp_path

importlib.reload(drive_only)

drive_only.drive_ramp(
    ramp_path(RAMP),
    PORT,
    output_root=PROJECT / 'LLM/runs/drive_only',
    confirm_start=False,
)

## 0 · 建立或继续 bench
重复运行本格会保留相同设置下的 session。当前继续本次实验；更换机位配置或开始独立实验时，将本格 SESSION_ID 改为 None。

In [ ]:
from importlib import reload
from LLM import bench as bench_module

SESSION_ID = None  # None 新建；恢复时填自己已有的 session ID
rotation = (False, False)
request = (SESSION_ID, tuple(CAMERAS), tuple(MODE), EXPOSURE_MS, GAIN, PORT, rotation)
if globals().get('_block0_request') != request or globals().get('_block0_bench') is not globals().get('bench'):
    Bench = reload(bench_module).Bench
    board = bench_module.BOARD
    W, H, FPS = MODE
    bench = Bench(cameras=CAMERAS, serial_port=PORT, width=W, height=H, fps=FPS,
                  exposure_ms=EXPOSURE_MS, gain=GAIN, upside_down=rotation,
                  board=board, resume_session=SESSION_ID)
    _block0_bench = bench
    _block0_request = request
    reference = bench.reference_pair
    AXIS_BACKGROUND_DIR = None
print('当前 session:', bench.folder.name)
print(f'双目标定与验证用板：{board.name}，格距 {board.square_mm:.2f} mm')


## 1 · 相机预览：对准、对焦、补光（可重跑）
画面直接显示在本格输出里（A | B），按「设置」里的 `MODE`、`EXPOSURE_MS`、`GAIN` 全分辨率、全帧率运行，不录像。自动曝光可能随照明改变实际帧率；以试录报告为准。每台相机显示平均帧率、最高帧率（最好的 0.5 s）和亮度。中断 kernel 结束预览。

- 默认自动曝光（EXPOSURE_MS=None）；每个相机入口都会显式切回自动曝光，不固定增益。若改为数值则使用新版手动控制。更改模式/曝光后从「建立 bench」重跑。
- 遮挡一个镜头确认 A/B。
- 调相机朝向、对焦和灯光，直到两机 mean ≈ 59/82、max ≈ 235；mean 低于 ~20 时检测不到板子。
- 目标为 MODE 中的 120 fps；若实际帧率明显偏低，检查曝光、补光和试录报告。
- `ROTATE_180`：每台相机单独转 180°，只能 0° 或 180°；其他角度靠转动相机本身。改变会**新建 session 并清空标定**：之后要设 `CALIBRATION = None` 重新标定，或载入在同一朝向下拍的标定。
- 标定之后再动相机或对焦，都要重新标定。

In [ ]:
# ---- 本格参数（曝光、增益、帧率在「设置」里） ----
ROTATE_180 = (False, False)  # 当前机位下两幅图均正立
SCALE = 1.0                 # 显示缩放，只影响显示大小（1.0 = 原始分辨率）

from importlib import reload
from controller.camera import uvc
reload(uvc)
bench.preview_cameras(rotate180=ROTATE_180, scale=SCALE)

In [ ]:
if CALIBRATION is None:
    # 正在运行的 notebook 也使用当前的标定预览；保留 bench 和本 session 的补拍记录。
    from importlib import reload
    from controller.camera import sources
    from controller.calib import capture
    reload(sources)
    reload(capture)
    from LLM.record.calibration import collect_appended_frames
    seed = None  # 可填同一 session 中已有的标定 images 目录
    combined = collect_appended_frames(bench, seed=seed)
    bench.calibrate_intrinsics(combined)
    bench.calibrate_extrinsics()
else:
    bench.use_calibration(CALIBRATION)

## 3 · 在最终机位验证（可重跑）
使用步骤 2 的同一块实物板，采集至少 5 个新的静止姿态，按 q 结束；须 PASS。

In [ ]:
from importlib import reload
from controller.calib import capture
reload(capture)
display(bench.verify_calibration())

## 4 · 参考照片（可重跑）
移走板子；LED 与垂线两机都要看得到。点 Save，再 Close。

In [ ]:
reference = bench.photograph_reference()
for tag in 'AB':
    display(Image(filename=str(reference/f'{tag}.png'), width=800))

## 4A · 查看台架方格中心（可重跑）
使用第 4 格保存的照片，`PLUMB = None` 时也可运行。青色十字标出候选中心，请看它是否落在台架方格的黑白交点上。A、B 各自编号，两图的同号点可能对应不同方格。

In [ ]:
from LLM.analyse.holder import inspect_holder

if bench.reference_pair is None:
    raise ValueError('请先在第 4 格保存本次 session 的 A、B 参考照片。')
regions = bench.reference_pair / 'holder_regions.json'
holder_preview = inspect_holder(bench.reference_pair, regions_path=regions if regions.exists() else None)
for tag in 'AB':
    view = holder_preview['cameras'][tag]
    print(f"相机 {tag} · {len(view['candidates'])} 个候选中心")
    display(Image(filename=view['overlay'], width=1000))

## 5 · 可选：装置定位（可重跑）
步骤见 `LLM/analyse/HOLDER_MARKERS.md`。`PLUMB = None` 时跳过，`run()` 做 sync check。

In [ ]:
if PLUMB is not None:
    display(bench.find_holder_marks())
    for tag in 'AB':
        display(Image(filename=str(reference/f'{tag}_holder_landmarks.png'), width=800))
    display(bench.locate_fixture(PLUMB))

## 6 · 10 秒试录（可重跑）
纯相机，自动审核。

## 6A · 可选：axis_pose 空背景（只用相机）
先移走 robot，保持相机、支架、曝光与照明不变。把 `CAPTURE_AXIS_BACKGROUND=True` 后单独运行，点 Save static reference pair，再 Close；随后放回 robot。
本格不改变步骤 4 的装置参考。换 session 后重新拍背景；已有背景也可在 9A 手动指定。

自动曝光下背景与 robot 帧可能亮度不同；若差分失检，先检查叠加图，再考虑显式 threshold 模式或为这组拍摄固定曝光。


In [ ]:
from LLM.record.panel import run_panel as _axis_panel

CAPTURE_AXIS_BACKGROUND = False
if CAPTURE_AXIS_BACKGROUND:
    CAPTURE_AXIS_BACKGROUND = False
    _axis_bg = _axis_panel(bench.session, output_root=bench.folder/'axis_background_takes', max_seconds=10)
    AXIS_BACKGROUND_DIR = _axis_bg['reference_pair']
    if AXIS_BACKGROUND_DIR is None: raise ValueError('请保存空背景双图再关闭预览')
    print('Axis pose 空背景:', AXIS_BACKGROUND_DIR)
    for tag in 'AB':
        display(Image(filename=str(Path(AXIS_BACKGROUND_DIR)/f'{tag}.png'), width=640))
else:
    print('未采集背景。已有空背景可在 Block 9A 指定。')


## 7 · 查看 ramp（可重跑）

In [ ]:
from io import BytesIO

with BytesIO() as ramp_png:
    bench.show_ramp(RAMP).savefig(ramp_png, format='png', bbox_inches='tight')
    display(Image(data=ramp_png.getvalue(), format='png'))

## 8 · 一键实验：自动录像 → 5 秒后启动 ramp + LED
运行本格后，程序自动打开双机并连接 ESP；收到 READY、取得双机画面后自动开始录像，无需在 Viser 点 Record 或 START。5 秒倒计时从录像开始计时，不是从点击 cell 计时。

- 先确认第 6 格试录通过、第 7 格曲线正确，停止其他相机预览。ESP reset 时保持线圈电源关闭；看到 READY 再开电源。
- 随时点本格 Stop / notebook Interrupt：倒计时中取消则不启动 ramp；驱动中中断则发送 STOP，等待最多约 1 秒确认 OFF，然后保存已有录像并释放相机、串口。Viser 的 Stop / finish 也保留。
- 正常跑完所选 ramp 后自动停止线圈，再录约 2 秒后收尾。不用点 Stop 也会结束。
- 等本格保存和审核结束后，可直接再次运行本格；每次新建录像目录，不覆盖上一条。相机设置和试录条件未变时，不必重跑第 0 格。
- 停止请使用 Interrupt，不用 Restart Kernel 或拔 USB 代替停止。


In [ ]:
from importlib import reload
from LLM import bench as bench_module
from LLM.record import panel

# 加载自动启动逻辑，沿用当前 session 和第 6 格的试录结果。
reload(panel)
run_experiment = reload(bench_module).Bench.run
display(run_experiment(bench, RAMP, note=NOTE, warnings_reviewed=WARNING_REVIEW, auto_start=True))

## 9 · 分析（可重跑）
LED 对时 → frequency + 四路 CS 曲线。

In [ ]:
display(bench.analyse())
for name in ('input_curves.png', 'overview.png'):
    if (bench.last_run/'export'/name).exists():
        display(Image(filename=str(bench.last_run/'export'/name)))

## 9A · 可选：axis_pose 离线检查
可直接分析步骤 8 已保存的录像，即使步骤 9 的 LED 对时失败也可独立运行。先完成匹配标定的验证。

填写两机 `ROBOT_ROIS`（保存图像中的 `[x,y,width,height]`，排除 LED/holder），核对空背景后设 `AXIS_BACKGROUND_CONFIRMED=True`。无背景需显式选 `threshold`。
将 `RUN_AXIS_POSE=True`，先保留 `AXIS_MAX_FRAMES=500` 检查分割叠加图；确认后改 `None` 处理全片。旧标定缺旋转元数据时，实际核对后才设置 `AXIS_PREPROCESSING_CONFIRMED=True`。

结果独立保存到 `processed/axis_pose_trials/`，是 camera A 下的实验性轴线/质心代理；时间从首对保存帧开始，不是 LED 零点，不用于正式 takeoff 或重力倾角。


In [ ]:
import importlib
import LLM.axis_pose as _axis_pose
importlib.reload(_axis_pose)

RUN_AXIS_POSE = False  # 填好下面参数后改 True，单独运行本格
AXIS_RUN = bench.last_run        # 也可填 Path('/.../LLM/runs/某次实验')
AXIS_RIG = (Path(AXIS_RUN)/'calibration/stereo_rig.json'
            if AXIS_RUN is not None and (Path(AXIS_RUN)/'calibration/stereo_rig.json').exists() else bench.calibration)
ROBOT_ROIS = {'A': None, 'B': None}  # 每台相机 [x, y, width, height]
AXIS_BACKGROUND_MODE = 'difference'  # 没有空背景可显式选 'threshold'
AXIS_BACKGROUND_PATH = globals().get('AXIS_BACKGROUND_DIR')  # 或填空背景双图目录
AXIS_BACKGROUND_CONFIRMED = False
AXIS_PREPROCESSING_CONFIRMED = False
AXIS_REFERENCE_CAMERA = (0., -1., 0.)
AXIS_THRESHOLD = 90
AXIS_MAX_FRAMES = 500  # 预览通过后改 None，处理完整录像

if RUN_AXIS_POSE:
    RUN_AXIS_POSE = False
    if AXIS_RUN is None: raise ValueError('先录完 Block 8，或指定已有 AXIS_RUN')
    if AXIS_RIG is None: raise ValueError('先完成步骤 2/3 的匹配标定，或显式指定 AXIS_RIG')
    _backgrounds = ({tag: Path(AXIS_BACKGROUND_PATH)/f'{tag}.png' for tag in 'AB'}
                    if AXIS_BACKGROUND_MODE == 'difference' and AXIS_BACKGROUND_PATH is not None else None)
    AXIS_RESULT = _axis_pose.run_axis_pose(
        AXIS_RUN, AXIS_RIG, robot_rois=ROBOT_ROIS,
        background_mode=AXIS_BACKGROUND_MODE, background_paths=_backgrounds,
        background_confirmed=AXIS_BACKGROUND_CONFIRMED,
        preprocessing_confirmed=AXIS_PREPROCESSING_CONFIRMED,
        reference_axis_camera=AXIS_REFERENCE_CAMERA, threshold=AXIS_THRESHOLD,
        max_frames=AXIS_MAX_FRAMES, progress=lambda done, total: print(f'axis_pose: {done}/{total}'))
    display({key: AXIS_RESULT[key] for key in ('status', 'frames', 'accepted', 'reasons', 'directory')})
    display(Image(filename=AXIS_RESULT['curves'], width=1000))
    display(Image(filename=AXIS_RESULT['overlays'], width=1100))
else:
    print('axis_pose 未运行；填好 ROI / 背景 / 标定后，将 RUN_AXIS_POSE 改 True。')
